# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gulzaibsharif-coder/flyrank-ml-capstone/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

This capstone asks whether page-level search-performance signals can be used to prioritize content for refresh before its search visibility declines. Using the FlyRank internship warehouse's `fact_content_daily_performance` table, a leakage-safe Random Forest classifier was trained on historical GSC/GA4 signals and compared against a majority-class baseline on a held-out, stratified split. Because the target (whether a page records a click) is imbalanced at roughly 8% positive, the baseline achieved higher raw accuracy (0.919 vs. 0.897), but the model provided non-zero precision, recall, and F1 (0.194 / 0.086 / 0.120) on the rare outcome that the baseline structurally cannot detect. An earlier version of the model showed inflated accuracy due to target leakage (via `gsc_clicks` and CTR), which was identified and removed before this final evaluation. The result is best read as a weak but honest decision-support signal for prioritizing pages for human review, not as strong predictive evidence or proof that refreshing content will improve rankings.

## 1. Question

### Research Question

Can page-level search-performance signals be used to identify content that should be prioritized for refresh before its search visibility deteriorates?

### Decision Supported

The analysis supports a content-refresh prioritization decision: which pages should be reviewed first based on measurable performance signals, rather than relying only on manual judgment.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Data

### Data

This study uses the FlyRank internship warehouse release containing the `fact_content_daily_performance` table. The dataset contains page-level daily search-performance observations used to study content-refresh prioritization.

The analysis uses the available reporting-period observations required by the capstone notebooks. Only fields necessary for the research question are retained.

The analysis excludes identifiers, client-specific information, private search queries, URLs, and other information that could expose confidential or personally identifiable information. Aggregations and derived features are used where possible to keep the published analysis public-safe.

The target and feature construction are separated carefully. Variables used directly to construct the target are not used as predictive features in the leakage-safe model.

### Data source

### Data source

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

### Public-safety rule

No client names, private URLs, private queries, or other confidential information are reproduced in the paper.

In [14]:
# Basic data-contract checks for the paper.
# The actual warehouse loading and transformation were completed
# in the preceding capstone notebooks.

print("Dataset: FlyRank internship warehouse")
print("Table: fact_content_daily_performance")
print("Purpose: page-level search-performance analysis")
print("Public-safety exclusions: client names, private URLs, private queries")


Dataset: FlyRank internship warehouse
Table: fact_content_daily_performance
Purpose: page-level search-performance analysis
Public-safety exclusions: client names, private URLs, private queries


## 3. Methodology

### Methodology

The analysis treats content-refresh prioritization as a supervised classification problem. The objective is not to predict search rankings directly, but to identify pages that warrant review based on observable search-performance signals.

### Assumptions

The analysis assumes that historical page-level performance signals contain useful information for prioritizing pages for review. This is a decision-support exercise rather than a causal estimate of the effect of refreshing content.

### Features

Candidate features are derived from page-level search-performance observations. Features are selected to represent measurable performance and change signals while avoiding variables that directly define the target.

### Label definition

The target represents whether an observation meets the predefined refresh-priority condition established during the capstone workflow.

The label is treated as a decision-support label rather than a ground-truth measure of whether a future content refresh will succeed.

### Baseline

A simple rule-based baseline is used as the reference point. The machine-learning model is evaluated against the same baseline and the same held-out evaluation data.

### Validation design

The final evaluation uses an unseen evaluation split. Where temporal information is available, observations are ordered by reporting period so that information from the future is not used to predict the past.

### Leakage checks

The analysis explicitly checks whether any feature is derived directly from the target or from information that would only be available after the prediction point.

In particular, variables used to construct a CTR-derived target are excluded from the predictive feature set. This prevents the model from learning the answer directly from the variables used to define the answer.

### Interpretation

Model output is interpreted as prioritization support. A high-priority page should be reviewed first; it should not be interpreted as proof that a content refresh will increase rankings, clicks, or traffic.

In [15]:
# Leakage-control checklist

target_name = "gsc_clicks_gt_0"  # target = (gsc_clicks > 0)

excluded_from_features = {
    "gsc_clicks": "Used directly to construct the target — including it leaks the answer.",
    "ctr": "Derived from gsc_clicks / gsc_impressions — inherits the same leakage.",
    "client_hash_id": "Identifier, not a predictive signal; also a public-safety exclusion.",
    "content_hash_id": "Identifier, not a predictive signal; also a public-safety exclusion.",
}

print("Target:", target_name)
print("Leakage-control exclusions:")
for feature, reason in excluded_from_features.items():
    print(f" - {feature}: {reason}")

print("\nValidation principle: evaluate on unseen data (80/20 stratified split).")
print("Interpretation: decision support, not causal prediction.")
print("See work/notebooks/w03_feature_leakage_check.ipynb for the correlation-based leakage test.")


Target: gsc_clicks_gt_0
Leakage-control exclusions:
 - gsc_clicks: Used directly to construct the target — including it leaks the answer.
 - ctr: Derived from gsc_clicks / gsc_impressions — inherits the same leakage.
 - client_hash_id: Identifier, not a predictive signal; also a public-safety exclusion.
 - content_hash_id: Identifier, not a predictive signal; also a public-safety exclusion.

Validation principle: evaluate on unseen data (80/20 stratified split).
Interpretation: decision support, not causal prediction.
See work/notebooks/w03_feature_leakage_check.ipynb for the correlation-based leakage test.


## 4. Results (vs baseline)
### Results

The final model is compared with the simple baseline on the same held-out evaluation split.

The comparison focuses on whether the model provides useful predictive separation beyond the baseline. Results are reported using the metrics produced by the leakage-safe evaluation rather than the earlier exploratory model (see `work/notebooks/w06_validation_audit.ipynb` for the full training and evaluation code).

The earlier modeling workflow demonstrated why leakage control matters: an earlier version of this model included `gsc_clicks` and CTR as features, even though the target is defined directly from `gsc_clicks`. This produced artificially inflated accuracy. The final analysis excludes these target-derived predictors and evaluates the model only on information that would be available at prediction time (see `work/notebooks/w03_feature_leakage_check.ipynb` for the leakage audit).

### Model versus baseline

| Approach | Evaluation metric | Result |
|---|---:|---:|
| Baseline rule (majority class) | Accuracy | **0.919** |
| Leakage-safe model | Accuracy | **0.897** |
| Leakage-safe model | Precision | **0.194** |
| Leakage-safe model | Recall | **0.086** |
| Leakage-safe model | F1 | **0.120** |

The target is imbalanced — approximately 8% of observations record a click. This means the majority-class baseline achieves a high accuracy simply by never predicting the minority class, at the cost of zero recall on the outcome the model is meant to help prioritize. The leakage-safe model trades a small amount of accuracy for the ability to identify some — though not most — of the true positive cases the baseline misses entirely.

The final result should be interpreted as evidence about predictive usefulness on the selected evaluation split, not as evidence that the model will produce the same performance on every future dataset or client.

### Main finding

**The baseline outperformed the leakage-safe model on raw accuracy (0.919 vs 0.897), but accuracy is a misleading metric here given the ~8% class imbalance.** On precision, recall, and F1 — the metrics that actually reflect ability to catch the rare "gets clicks" pages — the model provides a weak but non-zero signal that the baseline cannot provide at all, since a majority-class rule has zero recall by construction. This should be read as limited decision-support value rather than strong predictive power, and any prioritization use of the model's output should account for its low recall (roughly 1 in 12 true positives caught) and low precision (roughly 1 in 5 flagged pages actually being a true positive).

No stronger claim should be made without additional validation.

In [16]:
# Final results from the leakage-safe chronological evaluation.
# Baseline = majority-class rule on the same held-out test set.
# Final model = leakage-safe Random Forest.
# See work/notebooks/w06_validation_audit.ipynb for the full training/evaluation code.

baseline_accuracy = 0.919
model_accuracy = 0.897
model_precision = 0.1944
model_recall = 0.0864
model_f1 = 0.1197

results = {
    "Baseline accuracy": baseline_accuracy,
    "Leakage-safe model accuracy": model_accuracy,
    "Model precision": model_precision,
    "Model recall": model_recall,
    "Model F1": model_f1,
}

print("FINAL RESULTS")
for k, v in results.items():
    print(f"{k}: {v}")

print("\nMAIN FINDING:")
print("On raw accuracy, the baseline outperformed the leakage-safe model (0.919 vs 0.897).")
print("However, the target is imbalanced (~8% positive class), so accuracy alone is misleading.")
print("The model achieved non-zero precision/recall/F1 on the minority class, which the")
print("majority-class baseline cannot do at all (0 recall by construction).")

FINAL RESULTS
Baseline accuracy: 0.919
Leakage-safe model accuracy: 0.897
Model precision: 0.1944
Model recall: 0.0864
Model F1: 0.1197

MAIN FINDING:
On raw accuracy, the baseline outperformed the leakage-safe model (0.919 vs 0.897).
However, the target is imbalanced (~8% positive class), so accuracy alone is misleading.
The model achieved non-zero precision/recall/F1 on the minority class, which the
majority-class baseline cannot do at all (0 recall by construction).


## 5. Limitations
### Limitations & Honest Framing

This analysis has several important limitations.

First, the target is a decision-support label rather than a directly observed future business outcome. Therefore, a high predicted priority does not establish that refreshing a page will improve rankings, clicks, or traffic.

Second, the analysis is based on the available FlyRank internship dataset and its reporting periods. Results should not automatically be generalized to every website, industry, search environment, or future period.

Third, observational search-performance data cannot by itself establish causality. Associations between performance signals and refresh priority should not be interpreted as proof that changing a feature will cause a ranking improvement.

Fourth, model performance depends on the selected features, label definition, validation design, and evaluation period.

Fifth, the model's current predictive value is limited in practice — recall of 0.086 means it catches roughly 1 in 12 pages that actually receive clicks, and precision of 0.194 means about 1 in 5 flagged pages are true positives. As a standalone signal this is weak; it may improve with class-weighting, additional features (e.g. content-age, historical trend features), or a different target definition (e.g. predicting a meaningful *change* in performance rather than any click at all).

Finally, the earlier modeling workflow demonstrated the importance of leakage control. The final research artifact therefore uses a stricter feature definition and avoids features that directly encode the target.

### Honest claim

The strongest defensible claim is that the analysis provides a reproducible, data-informed framework for prioritizing content for human review. It does not establish that the model can guarantee future search-performance improvements.

In [17]:
limitations = [
    "Decision-support label is not a guaranteed future outcome.",
    "Results may not generalize to all websites or future periods.",
    "Observational data does not establish causality.",
    "Performance depends on feature, label, and validation design.",
    "Leakage control is required for trustworthy evaluation.",
]

for i, limitation in enumerate(limitations, start=1):
    print(f"{i}. {limitation}")


1. Decision-support label is not a guaranteed future outcome.
2. Results may not generalize to all websites or future periods.
3. Observational data does not establish causality.
4. Performance depends on feature, label, and validation design.
5. Leakage control is required for trustworthy evaluation.


## 6. Ranked recommendations
### Ranked Recommendations

The model output is translated into an action-oriented content-refresh playbook. Rankings should be treated as prioritization guidance for human review.

#### 1. Review the highest-priority pages first

Start with pages receiving the strongest refresh-priority signals. These pages represent the clearest candidates for investigation under the selected scoring framework.

**Action:** Review the page's recent search-performance trend, content quality, search intent alignment, and freshness before making an update.

#### 2. Investigate declining performance trends

Pages showing sustained deterioration should receive attention before pages with stable performance.

**Action:** Compare recent performance with the relevant historical period and determine whether the decline is persistent rather than a short-term fluctuation.

#### 3. Prioritize pages where improvement is actionable

A high model score alone should not trigger an automatic rewrite.

**Action:** Give higher operational priority to pages where the diagnosis suggests a concrete intervention, such as outdated information, weak intent alignment, incomplete coverage, or unclear content structure.

#### 4. Use human review before publishing changes

The model is a prioritization mechanism, not an autonomous content editor.

**Action:** Require a human reviewer to validate the recommendation and determine the appropriate content intervention.

#### 5. Measure the outcome after refresh

The framework should be used as part of an iterative measurement process.

**Action:** After a content update, compare subsequent performance with an appropriate pre-refresh period while avoiding claims that any observed change was caused solely by the refresh.

### Recommended operating sequence

**Detect → Diagnose → Prioritize → Human review → Refresh → Measure**

This sequence keeps the machine-learning output connected to an accountable editorial decision process.

In [18]:
recommendations = [
    "Review highest-priority pages first",
    "Investigate sustained performance declines",
    "Prioritize pages with actionable improvement opportunities",
    "Require human review before publishing changes",
    "Measure post-refresh performance",
]

print("RANKED ACTION PLAYBOOK")
for rank, recommendation in enumerate(recommendations, start=1):
    print(f"{rank}. {recommendation}")


RANKED ACTION PLAYBOOK
1. Review highest-priority pages first
2. Investigate sustained performance declines
3. Prioritize pages with actionable improvement opportunities
4. Require human review before publishing changes
5. Measure post-refresh performance


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Artifacts

The deployed research page embeds the main analytical artifacts needed to inspect the work:

1. Model-versus-baseline results table.
2. Evaluation metric comparison.
3. Model interpretation / feature importance where available.
4. Ranked content-refresh recommendations.
5. Reproducibility links to the notebooks and repository.

All published artifacts are checked for public safety before deployment. Private queries, client names, private URLs, credentials, tokens, and confidential identifiers must not appear in the published artifacts.


In [19]:
# Final artifact checklist

artifacts = [
    "Model versus baseline table",
    "Evaluation metric comparison",
    "Model interpretation / feature importance",
    "Ranked recommendations",
    "Reproducibility links",
]

print("PAPER ARTIFACT CHECKLIST")
for i, artifact in enumerate(artifacts, start=1):
    print(f"{i}. {artifact}")

PAPER ARTIFACT CHECKLIST
1. Model versus baseline table
2. Evaluation metric comparison
3. Model interpretation / feature importance
4. Ranked recommendations
5. Reproducibility links


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

# ML-12 — Tell the Story

## 5-Minute Demo Outline

### 0:00–0:45 — Question

The project addresses a practical content-refresh problem: when many pages have measurable search-performance signals, which pages should be reviewed first?

**Research question:** Can page-level search-performance signals be used to prioritize content for refresh before search visibility deteriorates?

The goal is decision support rather than automatic content editing or a guarantee of future ranking improvement.

### 0:45–1:45 — Method

The project uses page-level search-performance data from the FlyRank ML Internship dataset.

The workflow was:

1. Define a refresh-priority target.
2. Construct page-level performance features.
3. Establish a simple baseline.
4. Train a machine-learning classifier.
5. Evaluate the approach on held-out data.
6. Check for target leakage.
7. Translate the output into a ranked content-refresh playbook.

### 1:45–2:45 — One Chart

The main chart compares the machine-learning model with the baseline on the same evaluation split.

The chart is used to determine whether the model provides useful predictive information beyond the simple baseline.

### 2:45–3:45 — Honest Result

On raw accuracy, the majority-class baseline actually outperformed the leakage-safe model (0.919 vs 0.897). But the target is imbalanced — only about 8% of pages receive a click — so accuracy alone is misleading: the baseline gets its high score by never predicting the rare outcome, meaning it has zero recall.

The model, by contrast, achieved a precision of 0.194, recall of 0.086, and F1 of 0.120 — modest numbers, but non-zero, meaning it can surface some of the pages the baseline would miss entirely.

The earlier exploratory result also demonstrated why leakage checks are important: a model can appear extremely accurate when information used to construct the target is also supplied as a feature. Once that leakage was removed, the honest result is a weak but real signal — useful only as a decision-support aid, not as strong predictive evidence.

### 3:45–4:30 — Recommendation

The recommended operating sequence is:

**Detect → Diagnose → Prioritize → Human review → Refresh → Measure**

Pages receiving stronger priority signals should be reviewed first, but the model should not automatically trigger content changes.

### 4:30–5:00 — Closing

The project turns page-level search-performance signals into a reproducible framework for prioritizing content-review work.

The main takeaway is that useful machine-learning work requires not only a model, but also leakage controls, appropriate validation, careful interpretation, and a clear connection between model output and a real decision.

---

## Short Social Post

I built a machine-learning workflow for content-refresh prioritization using page-level search-performance data from the FlyRank ML Internship dataset.

The project asks a practical question: **which pages should be reviewed first when search-performance signals suggest that content may need attention?**

I developed a baseline comparison, built a machine-learning approach, checked for target leakage, evaluated the model on held-out data, and translated the output into an actionable workflow:

**Detect → Diagnose → Prioritize → Human review → Refresh → Measure**

The honest result: the baseline beat the model on raw accuracy, but the model captured signal the baseline structurally cannot (non-zero recall on a rare event). The biggest lesson was that building a model is only part of the job — reporting an honest, leakage-aware result mattered more than chasing an impressive-looking number.

#MachineLearning #DataScience #SEO #ContentStrategy #Research



---

## Employer-Facing Summary

I built a machine-learning content-refresh prioritization workflow using page-level search-performance data from the FlyRank ML Internship dataset. The project included feature engineering, baseline comparison, leakage checks, held-out evaluation, model interpretation, and translation of model output into a ranked content-review playbook. An early version of the model showed inflated accuracy due to target leakage; after correcting this, the honest result showed the model trailing the baseline on raw accuracy but providing non-zero precision/recall on a rare, imbalanced outcome that the baseline could not detect at all. The project produced a reproducible, leakage-aware decision-support framework while avoiding unsupported claims about model performance or future search outcomes.